In [38]:
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("../style/style-formal.mplstyle")

In [39]:
def calc_afl_perf_from_json(results, reproduced_bugs=None, ks=[1, 2, 3, 4, 5]):
    bug2min_afl_rank = dict()
    for bug_name in results["buggy_methods"]:
        if len(results["buggy_methods"][bug_name]) == 0:
            continue
        if reproduced_bugs is not None and bug_name not in reproduced_bugs:
            continue
        min_afl_rank = min(
            [
                v["autofl_rank"]
                for method_name, v in results["buggy_methods"][bug_name].items()
            ]
        )
        bug2min_afl_rank[bug_name] = min_afl_rank

    afl_perf = [
        len([v for bug_name, v in bug2min_afl_rank.items() if (v <= k)]) for k in ks
    ]
    return afl_perf

In [40]:
def calc_mrr(data):
    """
    Calculate MRR (Mean Reciprocal Rank)
    """
    total_rr = 0.0
    bug_cnt = 0

    for bug_name, method_info in data.get("buggy_methods", {}).items():
        if not method_info:
            continue
        
        # Collect ranks for all suspicious elements of this bug
        ranks = [
            info.get("autofl_rank")
            for info in method_info.values()
            if isinstance(info, dict) and info.get("autofl_rank") is not None
        ]

        if not ranks:
            continue

        best_rank = min(ranks)

        # reciprocal rank
        rr = 1.0 / best_rank

        total_rr += rr
        bug_cnt += 1

    if bug_cnt == 0:
        return 0.0

    return total_rr / bug_cnt


In [41]:
def calc_acc_at_n(data, n_rank=1):
    """
    Calculate Acc@N (Accuracy at N)
    Returns success bug count / total bug count
    """
    total_bug_cnt = 0
    success_cnt = 0

    for bug_name, method_info in data.get("buggy_methods", {}).items():
        if not method_info:
            continue

        total_bug_cnt += 1
        
        ranks = [
            info.get("autofl_rank")
            for info in method_info.values()
            if isinstance(info, dict) and info.get("autofl_rank") is not None
        ]
        
        if ranks:
            best_rank = min(ranks)
        else:
            continue
            
        if best_rank <= n_rank:
            success_cnt += 1

    if total_bug_cnt == 0:
        return 0.0, 0
    else:
        return round(success_cnt / total_bug_cnt, 3), success_cnt

In [42]:
def calc_ap(data):
    """
    Calculate Average Precision (AP)
    """
    result = {}

    for bug_name, method_info in data.get("buggy_methods", {}).items():
        if not method_info:
            continue
        
        result[bug_name] = []

        # Collect ranks for all suspicious elements of this bug
        ranks = [
            info.get("autofl_rank")
            for info in method_info.values()
            if isinstance(info, dict) and info.get("autofl_rank") is not None
        ]
        ranks.sort()
        for index, autofl_rank in enumerate(ranks):
            if autofl_rank is not None:
                val = (index + 1) / autofl_rank
                result[bug_name].append(val)

    total = 0
    cnt = 0
    for bug_name in result:
        ap_list = result[bug_name]
        total += sum(ap_list)
        cnt += len(ap_list)

    return total / cnt

In [43]:
def calc_map(data):
    """
    Compute Mean Average Precision (MAP)
    """
    ap_values = []

    for bug_name, method_info in data.get("buggy_methods", {}).items():
        if not method_info:
            continue

        # Collect absolute ranks of faulty methods
        ranks = [
            info.get("autofl_rank")
            for info in method_info.values()
            if isinstance(info, dict) and info.get("autofl_rank") is not None
        ]
        ranks.sort()

        # AP computation
        ap_sum = 0.0
        for idx, autofl_rank in enumerate(ranks, start=1):
            if autofl_rank is not None:
                ap_sum += (idx / autofl_rank)

        ap_i = ap_sum / len(ranks)
        ap_values.append(ap_i)

    if not ap_values:
        return 0.0

    return round(sum(ap_values) / len(ap_values), 6)

In [44]:
def main(exp_list, repetition=1, fig_title="Performance for GPT 4.1", legend_loc="lower right", bug_list=[]):
    score_data = {}
    for exp in exp_list:
        score_path_json = f"../combined_fl_results/{exp['name']}/{exp['gpt_version']}_R{repetition}_full_light.json"
        with open(score_path_json, "r") as f:
            data = json.load(f)
            data["buggy_methods"] = {k: v for k, v in data["buggy_methods"].items() if k in bug_list}
            score_data[exp['name']] = data
            
        map = round(calc_map(data), 3)
        mrr = round(calc_mrr(data), 3)


        result = {
            "label": exp["label"],
            "Mean Reciprocal Rank (MRR)": mrr,
            "Mean Average Precision (MAP)": map,
        }
        print(result)
        print()

In [45]:
import os

DATA_DIR = os.path.join(
    "../"
    "data/defects4j/",
)

# Looking for bugs with no reports in the last 30 days
no_report_bugs = []
for bug_name in sorted(os.listdir(DATA_DIR)):
    BUG_DIR = os.path.join(DATA_DIR, bug_name)
    current_bug_report = os.path.join(BUG_DIR, "current_report.json")
    target = os.path.join(BUG_DIR, "relevant_reports_timewindow_30.json")

    if not os.path.exists(current_bug_report) or os.path.exists(target):
        continue
    
    no_report_bugs.append(bug_name)

EXP_LIST = [
     {
        "name": "reportfl",
        "gpt_version": "gpt-4.1-mini-2025-04-14",
        "label": "None in 30days",
    },
]

main(EXP_LIST, repetition=5, bug_list=no_report_bugs)

{'label': 'None in 30days', 'Mean Reciprocal Rank (MRR)': 0.662, 'Mean Average Precision (MAP)': 0.604}



In [46]:
import os

DATA_DIR = os.path.join(
    "../"
    "data/defects4j/",
)

# Looking for bugs with reports in the last 30 days
no_report_bugs = []
for bug_name in sorted(os.listdir(DATA_DIR)):
    BUG_DIR = os.path.join(DATA_DIR, bug_name)
    current_bug_report = os.path.join(BUG_DIR, "current_report.json")
    target = os.path.join(BUG_DIR, "relevant_reports_timewindow_30.json")

    if not os.path.exists(current_bug_report) or not os.path.exists(target):
        continue
    
    no_report_bugs.append(bug_name)

EXP_LIST = [
     {
        "name": "reportfl",
        "gpt_version": "gpt-4.1-mini-2025-04-14",
        "label": "In 30days",
    },
]

main(EXP_LIST, repetition=5, bug_list=no_report_bugs)

{'label': 'In 30days', 'Mean Reciprocal Rank (MRR)': 0.728, 'Mean Average Precision (MAP)': 0.672}

